In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/paulasopala/Documents/Praktyki/smogcast"
)

SRC_PATH = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)

sys.path.insert(
    0,
    str(SRC_PATH),
)

print("Working directory:", Path.cwd())
print("src exists:", SRC_PATH.exists())
print("smogcast.db exists:", (PROJECT_ROOT / "smogcast.db").exists())

Working directory: /Users/paulasopala/Documents/Praktyki/smogcast
src exists: True
smogcast.db exists: True


In [2]:
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from smogcast.model.baseline import temporal_split
from smogcast.processing.features import build_features_from_db

print("Import smogcast OK")

Import smogcast OK


In [3]:
MODEL_PATH = "models/model_v2.joblib"
CUTOFF_DATE = "2025-08-20"

model_bundle = joblib.load(
    MODEL_PATH
)

rf_model = model_bundle["model"]
feature_columns = model_bundle["feature_columns"]

print(model_bundle.keys())
print(feature_columns)

dict_keys(['model', 'model_name', 'feature_columns', 'version', 'cutoff_date'])
['pm_lag_1d', 'pm_mean_3d', 'pm_mean_7d', 'month', 'day_of_week', 'is_weekend', 'is_heating_season', 'temp_c', 'wind_ms', 'humidity']


In [4]:
df = build_features_from_db()

print(df.shape)
print(df.head())

(34392, 19)
   station_id param_code       date  mean_value  max_value  coverage  \
0         117       PM10 2023-08-19        26.4       26.4  4.166667   
1         117       PM10 2023-08-20        18.1       18.1  4.166667   
2         117       PM10 2023-08-21        22.6       22.6  4.166667   
3         117       PM10 2023-08-22        18.7       18.7  4.166667   
4         117       PM10 2023-08-23        24.6       24.6  4.166667   

   pm_lag_1d  pm_mean_3d  pm_mean_7d  month  day_of_week  is_weekend  \
0        NaN         NaN         NaN      8            5        True   
1       26.4         NaN         NaN      8            6        True   
2       18.1         NaN         NaN      8            0       False   
3       22.6   22.366667         NaN      8            1       False   
4       18.7   19.800000         NaN      8            2       False   

   is_heating_season  target_value     temp_c   wind_ms   humidity  \
0              False          26.4        NaN       

In [5]:
train_df, test_df = temporal_split(
    df,
    cutoff_date=CUTOFF_DATE,
)

required_columns = feature_columns + [
    "target_value",
    "pm_lag_1d",
]

train_df = train_df.dropna(
    subset=required_columns
).copy()

test_df = test_df.dropna(
    subset=required_columns
).copy()

train_df["is_weekend"] = (
    train_df["is_weekend"].astype(int)
)

train_df["is_heating_season"] = (
    train_df["is_heating_season"].astype(int)
)

test_df["is_weekend"] = (
    test_df["is_weekend"].astype(int)
)

test_df["is_heating_season"] = (
    test_df["is_heating_season"].astype(int)
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 22788
Test: 11364


In [6]:
X_train = train_df[feature_columns]
y_train = train_df["target_value"]

X_test = test_df[feature_columns]
y_test = test_df["target_value"]

In [7]:
test_df["rf_prediction"] = (
    rf_model.predict(X_test)
)

In [8]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train,
)

test_df["linear_prediction"] = (
    linear_model.predict(X_test)
)

In [9]:
def calculate_metrics(
    y_true,
    y_pred,
):
    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = (
        mean_squared_error(
            y_true,
            y_pred,
        )
        ** 0.5
    )

    return {
        "mae": mae,
        "rmse": rmse,
    }


rf_metrics = calculate_metrics(
    y_test,
    test_df["rf_prediction"],
)

linear_metrics = calculate_metrics(
    y_test,
    test_df["linear_prediction"],
)

print("Random Forest")
print(rf_metrics)

print("\nLinear Regression")
print(linear_metrics)

Random Forest
{'mae': 5.595735852869789, 'rmse': 8.836330237870797}

Linear Regression
{'mae': 5.691217430683883, 'rmse': 8.46319497528673}


In [10]:
test_df["rf_error"] = (
    test_df["rf_prediction"]
    - test_df["target_value"]
)

test_df["linear_error"] = (
    test_df["linear_prediction"]
    - test_df["target_value"]
)

test_df["rf_absolute_error"] = (
    test_df["rf_error"].abs()
)

test_df["linear_absolute_error"] = (
    test_df["linear_error"].abs()
)

In [11]:
rf_threshold = (
    test_df["rf_absolute_error"]
    .quantile(0.95)
)

linear_threshold = (
    test_df["linear_absolute_error"]
    .quantile(0.95)
)

rf_worst = test_df[
    test_df["rf_absolute_error"]
    >= rf_threshold
]

linear_worst = test_df[
    test_df["linear_absolute_error"]
    >= linear_threshold
]

print(
    "RF - threshold for worst 5%:",
    round(rf_threshold, 3),
)

print(
    "RF - mean error in worst 5%:",
    round(
        rf_worst[
            "rf_absolute_error"
        ].mean(),
        3,
    ),
)

print()

print(
    "Linear - threshold for worst 5%:",
    round(linear_threshold, 3),
)

print(
    "Linear - mean error in worst 5%:",
    round(
        linear_worst[
            "linear_absolute_error"
        ].mean(),
        3,
    ),
)

RF - threshold for worst 5%: 17.132
RF - mean error in worst 5%: 27.376

Linear - threshold for worst 5%: 16.059
Linear - mean error in worst 5%: 25.355


In [12]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return "winter"

    if month in [3, 4, 5]:
        return "spring"

    if month in [6, 7, 8]:
        return "summer"

    return "autumn"


test_df["season"] = (
    test_df["month"]
    .apply(month_to_season)
)

winter_df = test_df[
    test_df["season"] == "winter"
].copy()

winter_rf = calculate_metrics(
    winter_df["target_value"],
    winter_df["rf_prediction"],
)

winter_linear = calculate_metrics(
    winter_df["target_value"],
    winter_df["linear_prediction"],
)

print("Winter observations:", len(winter_df))

print("\nRandom Forest - winter")
print(winter_rf)

print("\nLinear Regression - winter")
print(winter_linear)

Winter observations: 2861

Random Forest - winter
{'mae': 9.716068096257224, 'rmse': 14.168131828261146}

Linear Regression - winter
{'mae': 8.990275330592267, 'rmse': 12.905185805268822}


In [13]:
high_threshold = (
    test_df["target_value"]
    .quantile(0.90)
)

high_df = test_df[
    test_df["target_value"]
    >= high_threshold
].copy()

print(
    "High concentration threshold:",
    round(high_threshold, 3),
)

print(
    "Number of high observations:",
    len(high_df),
)

print(
    "RF mean signed error:",
    round(
        high_df["rf_error"].mean(),
        3,
    ),
)

print(
    "Linear mean signed error:",
    round(
        high_df["linear_error"].mean(),
        3,
    ),
)

print(
    "RF underestimates:",
    round(
        (
            high_df["rf_error"] < 0
        ).mean()
        * 100,
        1,
    ),
    "%",
)

print(
    "Linear underestimates:",
    round(
        (
            high_df["linear_error"] < 0
        ).mean()
        * 100,
        1,
    ),
    "%",
)

High concentration threshold: 37.233
Number of high observations: 1137
RF mean signed error: -8.264
Linear mean signed error: -11.428
RF underestimates: 76.5 %
Linear underestimates: 81.9 %


In [14]:
for param in [
    "PM10",
    "PM2.5",
]:
    param_df = test_df[
        test_df["param_code"] == param
    ].copy()

    high_threshold = (
        param_df["target_value"]
        .quantile(0.90)
    )

    high_param_df = param_df[
        param_df["target_value"]
        >= high_threshold
    ].copy()

    print(f"\n{param}")

    print(
        "High concentration threshold:",
        round(high_threshold, 3),
    )

    print(
        "Number of observations:",
        len(high_param_df),
    )

    print(
        "RF mean signed error:",
        round(
            high_param_df[
                "rf_error"
            ].mean(),
            3,
        ),
    )

    print(
        "Linear mean signed error:",
        round(
            high_param_df[
                "linear_error"
            ].mean(),
            3,
        ),
    )

    print(
        "RF underestimates:",
        round(
            (
                high_param_df[
                    "rf_error"
                ] < 0
            ).mean()
            * 100,
            1,
        ),
        "%",
    )

    print(
        "Linear underestimates:",
        round(
            (
                high_param_df[
                    "linear_error"
                ] < 0
            ).mean()
            * 100,
            1,
        ),
        "%",
    )


PM10
High concentration threshold: 41.71
Number of observations: 570
RF mean signed error: -10.671
Linear mean signed error: -13.595
RF underestimates: 79.3 %
Linear underestimates: 83.3 %

PM2.5
High concentration threshold: 31.542
Number of observations: 567
RF mean signed error: -5.302
Linear mean signed error: -8.878
RF underestimates: 72.0 %
Linear underestimates: 80.1 %


## Wnioski z analizy błędów

Random Forest osiąga niższe ogólne MAE, natomiast Linear Regression ma niższe RMSE oraz lepsze wyniki w okresie zimowym.

Analiza największych błędów pokazuje również, że Linear Regression lepiej radzi sobie z najbardziej skrajnymi pomyłkami.

Jednocześnie oba modele mają tendencję do niedoszacowywania wysokich stężeń.

Dla wysokich wartości PM10 Random Forest zaniża prognozę w około 79% przypadków, natomiast Linear Regression w około 83% przypadków.

Dla wysokich wartości PM2.5 Random Forest zaniża prognozę w około 72% przypadków, natomiast Linear Regression w około 80% przypadków.

Random Forest wykazuje więc mniejszą tendencję do niedoszacowywania wysokich stężeń niż Linear Regression.

Ze względu na niższe ogólne MAE oraz lepsze zachowanie przy wysokich stężeniach Random Forest pozostaje wybranym modelem produkcyjnym, mimo że Linear Regression osiąga niższe RMSE i lepsze wyniki zimą.